In [19]:
import config
import pandas as pd
import networkx as nx 
from networkx.algorithms.community import greedy_modularity_communities
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create undirected graph, save the degrees
G = nx.from_pandas_edgelist(
    pd.read_csv(config.EDGES_PATH),
    source="txId1",
    target="txId2",
    create_using=nx.Graph()
)

df_deg = pd.DataFrame(G.degree)

In [4]:
# Let's find comminities with a simple algorithm
communities = list(greedy_modularity_communities(G))
len(communities)

418

In [ ]:
# Now that we the communities let's map nodes to communities
nodes = {}

for i, comm in enumerate(communities):
    for node in comm:
        nodes[node] = i
        
df = pd.DataFrame({
    "txId": list(nodes.keys()),
    "community_id": list(nodes.values())
})

In [ ]:
# Add label for further analysis
df = df.merge(
    pd.read_csv(config.LABELS_PATH),
    on="txId",
    how="inner"
)
df["label"] = df["class"].map({"1":1, "2":0})

In [ ]:
# There are clearly big communities
# Note that the bad rate is not considering unknown transfers
community_summary = (
    df
    .groupby("community_id")
    .agg(
        n_nodes=("txId", "size"),
        n_illicit=("label", "sum"),
        n_licit=("label", lambda x: (x == 0).sum()),
        n_br=("label", "mean")
    )
).reset_index().fillna(0)
community_summary.sort_values("n_br").tail(10)

,community_id,n_nodes,n_illicit,n_licit,n_br
257,257,39,23.0,0,1.0
321,321,30,7.0,0,1.0
301,301,32,15.0,0,1.0
336,336,28,3.0,0,1.0
373,373,22,1.0,0,1.0
356,356,25,6.0,0,1.0
342,342,27,1.0,0,1.0
189,189,55,1.0,0,1.0
386,386,20,1.0,0,1.0
408,408,18,2.0,0,1.0


In [ ]:
df.merge(community_summary, on="community_id").to_csv("../data/data_communities.csv")